# 01 — NVIDIA Warp Basics for Colab

## 목표
이 notebook에서는 이후 mass–spring cube 실습에 필요한 Warp 문법만 빠르게 익힌다.

1. `@wp.kernel`과 `wp.launch`
2. `wp.tid()`
3. scalar / `wp.vec3` array
4. CPU vs CUDA 실행
5. Warp ↔ PyTorch zero-copy interoperability
6. 간단한 particle integration

> **수업 포인트**  
> Warp는 Python으로 kernel을 작성하지만, kernel 코드는 JIT 컴파일되어 CPU 또는 GPU에서 실행된다.
> 따라서 Python loop로 particle을 하나씩 업데이트하는 대신 수천~수백만 thread를 병렬 실행할 수 있다.

In [ ]:
# Colab 권장: Runtime > Change runtime type > T4 GPU
!nvidia-smi

# Warp 1.17.0의 PyPI Linux wheel은 CUDA 12.9 runtime 기반이라
# Colab의 일반적인 NVIDIA driver에서 호환성이 좋다.
%pip -q install "warp-lang==1.17.0"

import warp as wp
wp.init()
wp.print_diagnostics()

DEVICE = "cuda:0" if wp.is_cuda_available() else "cpu"
print("Selected Warp device:", DEVICE)

## 1. 첫 Warp kernel

각 thread가 배열의 원소 하나를 처리한다.

\[
y_i = 2x_i + 1
\]

In [ ]:
import numpy as np

@wp.kernel
def affine_kernel(x: wp.array(dtype=float),
                  y: wp.array(dtype=float)):
    i = wp.tid()
    y[i] = 2.0 * x[i] + 1.0

x_np = np.arange(10, dtype=np.float32)
x = wp.array(x_np, dtype=float, device=DEVICE)
y = wp.zeros(10, dtype=float, device=DEVICE)

wp.launch(affine_kernel, dim=10, inputs=[x, y], device=DEVICE)
wp.synchronize_device(DEVICE)

print("x =", x.numpy())
print("y =", y.numpy())

### 생각해보기
Python의 아래 loop와 Warp kernel은 수학적으로 같은 일을 한다.

```python
for i in range(N):
    y[i] = 2*x[i] + 1
```

하지만 Warp에서는 `dim=N`개의 logical thread가 이 작업을 나누어 수행한다.

## 2. `wp.vec3`: 3차원 particle state

mass–spring simulator에서는 위치, 속도, 힘을 3D vector로 저장한다.

In [ ]:
@wp.kernel
def gravity_kernel(force: wp.array(dtype=wp.vec3),
                   mass: float):
    i = wp.tid()
    force[i] = wp.vec3(0.0, 0.0, -9.81 * mass)

N = 8
force = wp.zeros(N, dtype=wp.vec3, device=DEVICE)

wp.launch(gravity_kernel, dim=N, inputs=[force, 0.15], device=DEVICE)
print(force.numpy())

## 3. Semi-implicit Euler integration

\[
v_{t+\Delta t} = v_t + a_t\Delta t
\]

\[
x_{t+\Delta t} = x_t + v_{t+\Delta t}\Delta t
\]

단순 explicit Euler보다 spring system에서 다루기 편한 기본 적분기로 사용한다.

In [ ]:
@wp.kernel
def integrate_kernel(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    force: wp.array(dtype=wp.vec3),
    mass: float,
    dt: float,
):
    i = wp.tid()

    a = force[i] / mass
    v_new = v[i] + a * dt
    x_new = x[i] + v_new * dt

    v[i] = v_new
    x[i] = x_new

x0 = np.zeros((8, 3), dtype=np.float32)
x0[:, 2] = 1.0

x = wp.array(x0, dtype=wp.vec3, device=DEVICE)
v = wp.zeros(8, dtype=wp.vec3, device=DEVICE)
f = wp.zeros(8, dtype=wp.vec3, device=DEVICE)

for step in range(100):
    wp.launch(gravity_kernel, dim=8, inputs=[f, 0.15], device=DEVICE)
    wp.launch(integrate_kernel, dim=8, inputs=[x, v, f, 0.15, 0.005], device=DEVICE)

print("z after falling:", x.numpy()[:, 2])

## 4. Warp ↔ PyTorch zero-copy

강화학습에서는 physics는 Warp, neural network는 PyTorch로 실행한다.

`wp.to_torch()`와 `wp.from_torch()`를 사용하면 같은 device memory를 공유할 수 있다.

In [ ]:
import torch

# Warp -> PyTorch
w = wp.array(np.arange(12, dtype=np.float32).reshape(4, 3),
             dtype=wp.vec3, device=DEVICE)

t = wp.to_torch(w)
print("Torch tensor:")
print(t)
print("device:", t.device)

# Torch tensor를 수정하면 같은 메모리를 공유하므로 Warp array에서도 보인다.
t += 10.0
wp.synchronize_device(DEVICE)
print("Warp view after Torch update:")
print(w.numpy())

In [ ]:
# PyTorch -> Warp
torch_device = torch.device("cuda" if DEVICE.startswith("cuda") else "cpu")
actions_torch = torch.zeros((32, 12), device=torch_device, dtype=torch.float32)

actions_warp = wp.from_torch(actions_torch, requires_grad=False)
print("Warp action shape:", actions_warp.shape)

## 5. 성능 비교용 kernel

첫 호출에는 JIT compile 시간이 포함된다. 따라서 timing은 warm-up 이후 측정한다.

In [ ]:
@wp.kernel
def saxpy_kernel(x: wp.array(dtype=float),
                 y: wp.array(dtype=float),
                 a: float):
    i = wp.tid()
    y[i] = a * x[i] + y[i]

N = 1_000_000
x = wp.ones(N, dtype=float, device=DEVICE)
y = wp.ones(N, dtype=float, device=DEVICE)

# warm-up
wp.launch(saxpy_kernel, dim=N, inputs=[x, y, 2.0], device=DEVICE)
wp.synchronize_device(DEVICE)

import time
tic = time.perf_counter()
for _ in range(100):
    wp.launch(saxpy_kernel, dim=N, inputs=[x, y, 2.0], device=DEVICE)
wp.synchronize_device(DEVICE)
toc = time.perf_counter()

print(f"100 launches: {toc-tic:.4f} s")

## Mini Exercise

1. `integrate_kernel`에 linear damping `-c v`를 추가하라.
2. `z < 0`일 때 바닥 penalty force를 넣어 particle이 바닥 아래로 떨어지지 않게 하라.
3. `N=8`, `N=2048*8`에서 실행 시간을 비교하라.

다음 notebook에서는 이 kernel들을 연결해 실제 cube를 만든다.